In [1]:
#Part 1: Imports

import os
import copy
import time
import random
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.preprocessing import (
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

In [2]:
#Part 2: Configuration
# =====================================================
# CONFIG
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", DEVICE)

MANIFEST = "/home/feliciano/dataset_manifest.csv"

GROUP_SPLIT = "/home/feliciano/group_split.csv"

BACKGROUND_FOLDER = (
    "/media/feliciano/Aux/"
    "AI_AFS_DATASET/"
    "AFS_BEHAVIOUR_DATASET/"
    "background"
)

SAMPLE_RATE = 16000

N_MELS = 128

FMAX = 4000

BATCH_SIZE = 32

EPOCHS = 50

LEARNING_RATE = 1e-3

PATIENCE = 10




Using device: cuda


In [3]:
#Part 3: Experiment B Dataset Loading

# =====================================================
# LOAD DATA
# =====================================================

fish_df = pd.read_csv(
    MANIFEST
)

split_df = pd.read_csv(
    GROUP_SPLIT
)

fish_df = fish_df.merge(
    split_df,
    on="parent_file_id",
    how="inner"
)

fish_df = fish_df[
    fish_df["label"]
    .str.lower()
    .isin(
        [
            "normal",
            "clustering",
            "agitation"
        ]
    )
].copy()

background_files = [

    os.path.join(
        BACKGROUND_FOLDER,
        f
    )

    for f in os.listdir(
        BACKGROUND_FOLDER
    )

    if f.lower().endswith(".wav")

]

background_df = pd.DataFrame({

    "clip_path":
        background_files,

    "label":
        "background"

})

bg_train, bg_test = train_test_split(

    background_df,

    test_size=0.20,

    random_state=SEED,

    shuffle=True

)

bg_train["split"] = "train"
bg_test["split"] = "test"

background_df = pd.concat(

    [
        bg_train,
        bg_test
    ],

    ignore_index=True

)

df = pd.concat(

    [
        fish_df,
        background_df
    ],

    ignore_index=True

)

In [4]:
#Part 4: Label Encoding

# =====================================================
# LABELS
# =====================================================

encoder = LabelEncoder()

df["label_encoded"] = encoder.fit_transform(
    df["label"]
)

print("\nClasses:")

for i, cls in enumerate(
    encoder.classes_
):
    print(i, cls)

    


Classes:
0 agitation
1 background
2 clustering
3 normal


In [7]:
#Part 5: Train/Test Split
# =====================================================
# TRAIN TEST SPLIT
# =====================================================

train_df = df[
    df["split"] == "train"
].copy()

test_df = df[
    df["split"] == "test"
].copy()


In [8]:
#Part 6: Validation Split

# =====================================================
# VALIDATION SPLIT
# =====================================================

gss = GroupShuffleSplit(

    n_splits=1,

    test_size=0.10,

    random_state=SEED

)

train_idx, val_idx = next(

    gss.split(

        train_df,

        groups=np.arange(
            len(train_df)
        )

    )

)

train_final = train_df.iloc[
    train_idx
]

val_final = train_df.iloc[
    val_idx
]

print("Training:", len(train_final))
print("Validation:", len(val_final))
print("Testing:", len(test_df))

Training: 15479
Validation: 1720
Testing: 4500


In [9]:
#Part 7: Mel Spectrogram Extraction

# =====================================================
# MEL EXTRACTION
# =====================================================

def extract_mel(path):

    signal, sr = librosa.load(

        path,

        sr=SAMPLE_RATE,

        mono=True

    )

    mel = librosa.feature.melspectrogram(

        y=signal,

        sr=sr,

        n_mels=N_MELS,

        fmax=FMAX

    )

    mel_db = librosa.power_to_db(

        mel,

        ref=np.max

    )

    return mel_db.astype(
        np.float32
    )

In [10]:
#Part 8: MelDataset
# =====================================================
# DATASET
# =====================================================

class MelDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe

    def __len__(self):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        mel = extract_mel(
            row["clip_path"]
        )

        mel = torch.tensor(

            mel,

            dtype=torch.float32

        ).unsqueeze(0)

        label = torch.tensor(

            row["label_encoded"],

            dtype=torch.long

        )

        return (
            mel,
            label
        )


In [11]:
#Part 9: DataLoaders
# =====================================================
# DATALOADERS
# =====================================================

train_ds = MelDataset(
    train_final
)

val_ds = MelDataset(
    val_final
)

test_ds = MelDataset(
    test_df
)

train_dl = DataLoader(

    train_ds,

    batch_size=BATCH_SIZE,

    shuffle=True

)

val_dl = DataLoader(

    val_ds,

    batch_size=BATCH_SIZE,

    shuffle=False

)

test_dl = DataLoader(

    test_ds,

    batch_size=BATCH_SIZE,

    shuffle=False

)

print(
    "\nTraining samples:",
    len(train_ds)
)

print(
    "Validation samples:",
    len(val_ds)
)

print(
    "Testing samples:",
    len(test_ds)
)



Training samples: 15479
Validation samples: 1720
Testing samples: 4500


In [12]:
#Part 10: Class Weights

# =====================================================
# CLASS WEIGHTS
# =====================================================

class_counts = np.bincount(

    train_final[
        "label_encoded"
    ]

)

print(
    "\nClass counts:"
)

print(
    class_counts
)

weights = (

    len(train_final)

    /

    (

        len(class_counts)

        * class_counts

    )

)

weights = np.sqrt(
    weights
)

weights = torch.tensor(

    weights,

    dtype=torch.float32

).to(
    DEVICE
)

print(
    "\nClass weights:"
)

print(
    weights
)



Class counts:
[2461 1079 4371 7568]

Class weights:
tensor([1.2540, 1.8938, 0.9409, 0.7151], device='cuda:0')


In [13]:
# Part 11 Original CNN=======================================
# ORIGINAL FISHCNN
# =====================================================

class FishCNN(nn.Module):

    def __init__(
        self,
        num_classes=4
    ):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                2
            ),

            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                2
            ),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                2
            )

        )

        self.pool = nn.AdaptiveAvgPool2d(
            (4,4)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64*4*4,
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.5
            ),

            nn.Linear(
                128,
                num_classes
            )

        )

    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )

        x = self.pool(
            x
        )

        x = self.classifier(
            x
        )

        return x

In [14]:
#Part 12: Model Definition

# =====================================================
# MODEL
# =====================================================
model = FishCNN(
    num_classes=4
).to(DEVICE)


criterion = nn.CrossEntropyLoss(

    weight=weights,

    label_smoothing=0.1

)

optimizer = optim.Adam(

    model.parameters(),

    lr=LEARNING_RATE

)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=3

)

print("\nModel Initialised")

print(
    f"Classes: {len(encoder.classes_)}"
)

print(
    f"Device: {DEVICE}"
)




Model Initialised
Classes: 4
Device: cuda


In [15]:
#Part 13: Training Loop

# =====================================================
# TRAINING
# =====================================================

best_f1 = 0.0

best_weights = None

patience_counter = 0

start_time = time.time()

print("\nTraining...\n")

for epoch in range(EPOCHS):

    # -----------------------------------------
    # TRAIN
    # -----------------------------------------

    model.train()

    running_loss = 0.0

    for mel, label in train_dl:

        mel = mel.to(
            DEVICE
        )

        label = label.to(
            DEVICE
        )

        optimizer.zero_grad()

        outputs = model(
            mel
        )

        loss = criterion(

            outputs,

            label

        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = (
        running_loss
        /
        len(train_dl)
    )

    # -----------------------------------------
    # VALIDATION
    # -----------------------------------------

    model.eval()

    val_true = []

    val_pred = []

    with torch.no_grad():

        for mel, label in val_dl:

            mel = mel.to(
                DEVICE
            )

            outputs = model(
                mel
            )

            preds = torch.argmax(

                outputs,

                dim=1

            )

            val_true.extend(
                label.numpy()
            )

            val_pred.extend(
                preds.cpu().numpy()
            )

    val_f1 = f1_score(

        val_true,

        val_pred,

        average="macro"

    )

    scheduler.step(
        val_f1
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(

        f"Epoch {epoch+1:02d}/{EPOCHS} | "

        f"Loss={train_loss:.4f} | "

        f"Val_F1={val_f1:.4f} | "

        f"LR={current_lr:.6f}"

    )

    # -----------------------------------------
    # SAVE BEST MODEL
    # -----------------------------------------

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_weights = copy.deepcopy(

            model.state_dict()

        )

        patience_counter = 0

    else:

        patience_counter += 1

    # -----------------------------------------
    # EARLY STOPPING
    # -----------------------------------------

    if patience_counter >= PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break

training_time = (

    time.time()

    - start_time

)

print(
    f"\nTraining completed in {training_time:.2f} seconds"
)


Training...

Epoch 01/50 | Loss=1.1683 | Val_F1=0.6180 | LR=0.001000
Epoch 02/50 | Loss=0.9354 | Val_F1=0.6343 | LR=0.001000
Epoch 03/50 | Loss=0.8603 | Val_F1=0.7966 | LR=0.001000
Epoch 04/50 | Loss=0.8241 | Val_F1=0.7970 | LR=0.001000
Epoch 05/50 | Loss=0.7930 | Val_F1=0.8394 | LR=0.001000
Epoch 06/50 | Loss=0.7680 | Val_F1=0.8410 | LR=0.001000
Epoch 07/50 | Loss=0.7483 | Val_F1=0.8474 | LR=0.001000
Epoch 08/50 | Loss=0.7361 | Val_F1=0.8559 | LR=0.001000
Epoch 09/50 | Loss=0.7250 | Val_F1=0.8695 | LR=0.001000
Epoch 10/50 | Loss=0.7148 | Val_F1=0.8710 | LR=0.001000
Epoch 11/50 | Loss=0.7032 | Val_F1=0.8802 | LR=0.001000
Epoch 12/50 | Loss=0.6947 | Val_F1=0.8832 | LR=0.001000
Epoch 13/50 | Loss=0.6886 | Val_F1=0.8867 | LR=0.001000
Epoch 14/50 | Loss=0.6792 | Val_F1=0.8942 | LR=0.001000
Epoch 15/50 | Loss=0.6726 | Val_F1=0.8887 | LR=0.001000
Epoch 16/50 | Loss=0.6700 | Val_F1=0.8818 | LR=0.001000
Epoch 17/50 | Loss=0.6647 | Val_F1=0.8941 | LR=0.001000
Epoch 18/50 | Loss=0.6587 | Val_F1

In [16]:
#Part 14: Load Best Model + Testing + Metrics

# =====================================================
# LOAD BEST MODEL
# =====================================================

print(
    "\nLoading best model..."
)

model.load_state_dict(
    best_weights
)

# =====================================================
# TESTING
# =====================================================

model.eval()

y_true = []
y_pred = []

start_inf = time.time()

with torch.no_grad():

    for mel, label in test_dl:

        mel = mel.to(
            DEVICE
        )

        outputs = model(
            mel
        )

        preds = torch.argmax(

            outputs,

            dim=1

        )

        y_true.extend(
            label.numpy()
        )

        y_pred.extend(
            preds.cpu().numpy()
        )

inference_time = (

    time.time()

    - start_inf

) / len(test_ds)

# =====================================================
# METRICS
# =====================================================

acc = accuracy_score(

    y_true,

    y_pred

)

prec = precision_score(

    y_true,

    y_pred,

    average="macro"

)

rec = recall_score(

    y_true,

    y_pred,

    average="macro"

)

f1 = f1_score(

    y_true,

    y_pred,

    average="macro"

)

cm = confusion_matrix(

    y_true,

    y_pred

)

print(
    "\n================================"
)

print(
    "ORIGINAL FISHCNN BACKGROUND"
)

print(
    "================================"
)

print(
    f"Accuracy : {acc:.4f}"
)

print(
    f"Precision: {prec:.4f}"
)

print(
    f"Recall   : {rec:.4f}"
)

print(
    f"F1       : {f1:.4f}"
)

print(
    f"Best Val : {best_f1:.4f}"
)

print(
    f"Training : {training_time:.2f}s"
)

print(
    f"Inference: {inference_time*1000:.4f} ms/sample"
)

print(
    "\nClassification Report\n"
)

print(

    classification_report(

        y_true,

        y_pred,

        target_names=
        encoder.classes_

    )

)



Loading best model...

ORIGINAL FISHCNN BACKGROUND
Accuracy : 0.7349
Precision: 0.7008
Recall   : 0.7596
F1       : 0.7165
Best Val : 0.9270
Training : 4348.38s
Inference: 5.0919 ms/sample

Classification Report

              precision    recall  f1-score   support

   agitation       0.27      0.52      0.36       390
  background       1.00      1.00      1.00       300
  clustering       0.65      0.80      0.72       870
      normal       0.88      0.72      0.79      2940

    accuracy                           0.73      4500
   macro avg       0.70      0.76      0.72      4500
weighted avg       0.79      0.73      0.75      4500



In [17]:
#Next: Part 15: Save Outputs

# =====================================================
# SAVE PREDICTIONS
# =====================================================

pred_df = pd.DataFrame({

    "clip_path":
        test_df[
            "clip_path"
        ].values,

    "true_label":
        encoder.inverse_transform(
            np.array(y_true)
        ),

    "predicted_label":
        encoder.inverse_transform(
            np.array(y_pred)
        )

})

pred_df.to_csv(

    "fishcnn_original_background_predictions.csv",

    index=False

)

# =====================================================
# SAVE CONFUSION MATRIX
# =====================================================

pd.DataFrame(

    cm,

    index=encoder.classes_,

    columns=encoder.classes_

).to_csv(

    "fishcnn_original_background_cm.csv"

)

# =====================================================
# SAVE PER-CLASS F1
# =====================================================

per_class_f1 = f1_score(

    y_true,

    y_pred,

    average=None

)

pd.DataFrame({

    "class":
        encoder.classes_,

    "f1":
        per_class_f1

}).to_csv(

    "fishcnn_original_background_per_class_f1.csv",

    index=False

)

# =====================================================
# SAVE MODEL
# =====================================================

torch.save(

    model.state_dict(),

    "fishcnn_original_background.pth"

)

# =====================================================
# SAVE LABEL ENCODER
# =====================================================

joblib.dump(

    encoder,

    "fishcnn_original_background_encoder.pkl"

)

# =====================================================
# FINISHED
# =====================================================

print("\n================================")
print("FILES SAVED")
print("================================")

print(
    "fishcnn_original_background.pth"
)

print(
    "fishcnn_original_background_encoder.pkl"
)

print(
    "fishcnn_original_background_predictions.csv"
)

print(
    "fishcnn_original_background_cm.csv"
)

print(
    "fishcnn_original_background_per_class_f1.csv"
)

print("\nDone")

NameError: name 'joblib' is not defined